# Experiment 46B — Solar DLinear + OOF Historical Memory: All Horizons

This experiment evaluates **Solar, H in {96, 192, 336, 720}** with the same frozen-retriever and chronological OOF integration protocol used by the H=96 pilot. A completed H=96 condition is detected and reused automatically. Only the direct backbone is changed to the official Time-Series-Library DLinear implementation.

The custom Solar split is retained exactly: 8,640 train, 2,880 validation, and the remaining 5,900 observations for test. No target-test hyperparameter tuning is performed.


In [ ]:

from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext

import gc
import importlib
import math
import os
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings(
    "ignore"
)

pd.set_option(
    "display.max_columns",
    260,
)

pd.set_option(
    "display.width",
    560,
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASET = "Solar"
EXPECTED_FULL_CHANNELS = 137

BACKBONES = [
    "DLinear",
]

HORIZONS = [
    96,
    192,
    336,
    720,
]

RET_SEQ_LEN = 96

TOP_K = 10
MEMORY_STRIDE = 8
OOF_ANCHOR_STRIDE = 4

FOLDS = [
    (
        0.55,
        0.70,
    ),
    (
        0.70,
        0.85,
    ),
    (
        0.85,
        1.00,
    ),
]

# Frozen predictive representation.
REP_PATCH_LEN = 16
REP_PATCH_STRIDE = 16
REP_D_MODEL = 64
REP_N_HEADS = 4
REP_LAYERS = 2
REP_D_FF = 128
REP_DROPOUT = 0.1
REP_DIM = 64

REP_NUM_PATCHES = (
    1
    + (
        RET_SEQ_LEN
        - REP_PATCH_LEN
    )
    // REP_PATCH_STRIDE
)

# Prospectively frozen Solar retriever training.
# These values are declared before any Solar test metric is evaluated.
RETRIEVER_CANDIDATE_M = 100
RETRIEVER_TAU = 0.5
RETRIEVER_LR = 1e-3
RETRIEVER_WD = 1e-4
RETRIEVER_BATCH_QUERIES = 32
RETRIEVER_MAX_EPOCHS = 30
RETRIEVER_PATIENCE = 6
RETRIEVER_GRAD_CLIP = 5.0
RETRIEVER_QUERY_STRIDE = 4
RETRIEVER_MEMORY_FRACTION = 0.60
RETRIEVER_PHASEA_TRAIN_FRACTION = 0.80
RETRIEVER_SEED = 0

# Frozen gate.
GATE_DIM = 26
GATE_LR = 1e-3
GATE_WD = 1e-4
GATE_BATCH = 8192
GATE_MAX_EPOCHS = 50
GATE_PATIENCE = 7

ALPHA_GRID = np.round(
    np.arange(
        0.0,
        1.0001,
        0.1,
    ),
    10,
)

LAMBDA_GRID = np.array(
    [
        0.0,
        0.25,
        0.50,
        0.75,
        1.00,
    ],
    dtype=np.float32,
)

TARGET_RETRIEVAL_PAIRS = 672

EPS = 1e-8

RETRIEVER_USE_AMP = (
    torch.cuda.is_available()
)

CROSSFIT_SEED = 2828

RESUME = True
FORCE = False

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_BLOCK_LEN = 24
BOOTSTRAP_SEED = 282800

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "solar_dlinear_oof_historical_memory"
)

ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SHARED_MEMORY_EMB_DIR = (
    ROOT
    / "shared_memory_embeddings"
)

SHARED_MEMORY_EMB_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACT_DIR = (
    ROOT
    / "artifacts"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUMMARY_PATH = (
    ROOT
    / "summary.csv"
)

BOOTSTRAP_PATH = (
    ROOT
    / "bootstrap.csv"
)

CURRENT_BACKBONE = None
DIRS = None

# Conservative direct inference blocks.
DIRECT_ANCHOR_BLOCK = {
    "PatchTST":
        16,
    "iTransformer":
        16,
    "TimeMixer":
        16,
    "DLinear":
        256,
}

print(
    "Device:",
    DEVICE,
)

print(
    "Output:",
    ROOT,
)


In [ ]:
SOLAR_CANDIDATES = [
    Path("/data/Time-Series-Library/dataset/Solar/solar_AL.txt"),
    Path("/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt"),
    Path("/data/pcw_workspace/Time-Series-Library/dataset/Solar/solar_AL.txt"),
    Path("/code/Time-Series-Library/dataset/Solar/solar_AL.txt"),
    Path("/data/dataset/Solar/solar_AL.txt"),
    Path("/data/dataset/solar_AL.txt"),
]

SOLAR_PATH = next(
    (p for p in SOLAR_CANDIDATES if p.is_file()),
    None,
)

if SOLAR_PATH is None:
    print("Attempted Solar paths:")
    for p in SOLAR_CANDIDATES:
        print(" -", p)
    raise FileNotFoundError("Could not find Solar-Energy solar_AL.txt.")


def load_solar_txt(path):
    numeric = pd.read_csv(path, header=None)
    if numeric.shape[1] == 1:
        numeric = pd.read_csv(path, header=None, sep=r"\s+")
    numeric = numeric.apply(pd.to_numeric, errors="coerce")
    numeric = numeric.dropna(axis=1, how="all")
    numeric = (
        numeric
        .replace([np.inf, -np.inf], np.nan)
        .interpolate(axis=0, limit_direction="both")
        .ffill()
        .bfill()
    )
    return numeric


raw_df = load_solar_txt(SOLAR_PATH)
raw = raw_df.to_numpy(dtype=np.float32)
n_time, n_channels = raw.shape

if n_time != 52560:
    raise ValueError(
        f"Standard Solar expects 52,560 rows, loaded {n_time}."
    )

if n_channels != EXPECTED_FULL_CHANNELS:
    raise ValueError(
        f"Solar expects exactly {EXPECTED_FULL_CHANNELS} numeric channels, "
        f"loaded {n_channels}."
    )

# Exact chronological 70/10/20 protocol used by the project.
train_end = 36792
val_end = 42048
test_end = n_time

if test_end != 52560:
    raise ValueError(f"Unexpected Solar length: {test_end}")

train_mean = raw[:train_end].mean(axis=0).astype(np.float32)
train_std = raw[:train_end].std(axis=0, ddof=0).astype(np.float32)

degenerate = np.where(train_std <= 1e-6)[0]

if len(degenerate):
    raise ValueError(
        "Solar contains train-degenerate channels: "
        f"{degenerate.tolist()}"
    )

z_full = (
    (raw - train_mean[None, :])
    / train_std[None, :]
).astype(np.float32)

marks = np.zeros((n_time, 0), dtype=np.float32)

DATA = {
    "Solar": {
        "name": "Solar",
        "path": SOLAR_PATH,
        "raw": raw,
        "z": z_full,
        "marks": marks,
        "n_channels": n_channels,
        "raw_channel_indices": np.arange(n_channels, dtype=np.int64),
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
    }
}

display(
    pd.DataFrame([
        {
            "Split": "Train",
            "Start": 0,
            "EndExclusive": train_end,
            "Length": train_end,
        },
        {
            "Split": "Validation",
            "Start": train_end,
            "EndExclusive": val_end,
            "Length": val_end - train_end,
        },
        {
            "Split": "Test",
            "Start": val_end,
            "EndExclusive": test_end,
            "Length": test_end - val_end,
        },
    ])
)

print("Solar path:", SOLAR_PATH)
print("Shape:", raw.shape)
print("Columns:", list(raw_df.columns))
print("train_end / val_end / test_end:", train_end, val_end, test_end)


In [ ]:

def prefix_normalize(
    raw_array,
    prefix,
):
    prefix = int(
        prefix
    )

    mean = raw_array[
        :prefix
    ].mean(
        axis=0,
    ).astype(
        np.float32
    )

    std = raw_array[
        :prefix
    ].std(
        axis=0,
        ddof=0,
    ).astype(
        np.float32
    )

    if np.any(
        std
        <= 1e-6
    ):
        raise ValueError(
            f"Degenerate prefix channel at prefix={prefix}."
        )

    z = (
        (
            raw_array
            - mean[
                None,
                :
            ]
        )
        / std[
            None,
            :
        ]
    ).astype(
        np.float32
    )

    return (
        z,
        {
            "Mean":
                mean,
            "Std":
                std,
            "Prefix":
                prefix,
        },
    )


def eval_anchors(
    start,
    end,
    horizon,
    stride=1,
    lookback=RET_SEQ_LEN,
):
    return np.arange(
        max(
            int(
                start
            ),
            int(
                lookback
            ),
        ),
        int(
            end
        )
        - int(
            horizon
        )
        + 1,
        int(
            stride
        ),
        dtype=np.int64,
    )


In [ ]:

BASE = Path(
    "/code/stock_regime_retrieval/"
    "strong_forecaster"
)

REPO_CANDIDATES = {
    "DLinear": [
        Path("/data/Time-Series-Library_v2"),
        Path("/data/Time-Series-Library"),
        Path("/data/pcw_workspace/Time-Series-Library"),
        Path("/code/Time-Series-Library"),
        BASE / "Time-Series-Library",
    ],
    "PatchTST": [
        BASE
        / "PatchTST_official",
        Path(
            "/code/PatchTST_official"
        ),
        Path(
            "/data/PatchTST_official"
        ),
    ],

    "iTransformer": [
        BASE
        / "iTransformer_official",
        Path(
            "/code/iTransformer"
        ),
        Path(
            "/data/iTransformer"
        ),
    ],

    "TimeMixer": [
        BASE
        / "TimeMixer_official",
        Path(
            "/code/TimeMixer"
        ),
        Path(
            "/data/TimeMixer"
        ),
    ],
}

REPO_URLS = {
    "DLinear":
        "https://github.com/thuml/Time-Series-Library.git",
    "PatchTST":
        "https://github.com/yuqinie98/PatchTST.git",
    "iTransformer":
        "https://github.com/thuml/iTransformer.git",
    "TimeMixer":
        "https://github.com/kwuking/TimeMixer.git",
}

EXPECTED_FILES = {
    "DLinear":
        Path("models/DLinear.py"),
    "PatchTST":
        Path(
            "PatchTST_supervised/"
            "models/PatchTST.py"
        ),
    "iTransformer":
        Path(
            "model/"
            "iTransformer.py"
        ),
    "TimeMixer":
        Path(
            "models/"
            "TimeMixer.py"
        ),
}

REPOS = {}

for backbone in BACKBONES:
    repo = next(
        (
            p
            for p in REPO_CANDIDATES[
                backbone
            ]
            if (
                p
                / EXPECTED_FILES[
                    backbone
                ]
            ).is_file()
        ),
        None,
    )

    if repo is None:
        repo = REPO_CANDIDATES[
            backbone
        ][
            0
        ]

        repo.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        print(
            f"Cloning {backbone} -> {repo}"
        )

        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                REPO_URLS[
                    backbone
                ],
                str(
                    repo
                ),
            ],
            check=True,
        )

    REPOS[
        backbone
    ] = repo

repo_rows = []

for backbone, repo in REPOS.items():
    try:
        commit = subprocess.check_output(
            [
                "git",
                "-C",
                str(
                    repo
                ),
                "rev-parse",
                "HEAD",
            ],
            text=True,
        ).strip()
    except Exception:
        commit = "unknown"

    repo_rows.append({
        "Backbone":
            backbone,
        "Repository":
            str(
                repo
            ),
        "Commit":
            commit,
    })

display(
    pd.DataFrame(
        repo_rows
    )
)

pd.DataFrame(
    repo_rows
).to_csv(
    ARTIFACT_DIR
    / "backbone_repository_commits.csv",
    index=False,
)


In [ ]:
DIRECT_RECIPES = {
    "DLinear": {
        "seq_len": 96,
        "label_len": 48,
        "batch_size": 32,
        "eval_batch": 256,
        "train_epochs": 10,
        "patience": 3,
        "learning_rate": 5e-3,
        "weight_decay": 0.0,
        "scheduler": "type1",
        "seed": 2021,
        "moving_avg": 25,
        "individual": False,
    },
}

display(pd.DataFrame(DIRECT_RECIPES).T)


In [ ]:

ACTIVE_MODEL_CLASS = None
ACTIVE_REPO = None


def clear_forecasting_modules():
    prefixes = [
        "models",
        "model",
        "layers",
        "utils",
        "data_provider",
        "exp",
        "experiments",
    ]

    for module_name in list(
        sys.modules.keys()
    ):
        if any(
            module_name
            == p
            or module_name.startswith(
                p
                + "."
            )
            for p in prefixes
        ):
            del sys.modules[
                module_name
            ]


def activate_backbone(
    backbone,
):
    global ACTIVE_MODEL_CLASS
    global ACTIVE_REPO

    clear_forecasting_modules()

    # Remove all known forecasting repositories from sys.path.
    repo_paths = []

    for b, r in REPOS.items():
        if b == "PatchTST":
            repo_paths.append(
                str(
                    r
                    / "PatchTST_supervised"
                )
            )
        else:
            repo_paths.append(
                str(
                    r
                )
            )

    sys.path[:] = [
        p
        for p in sys.path
        if p not in repo_paths
    ]

    if backbone == "DLinear":
        root = REPOS[backbone]
        sys.path.insert(0, str(root))
        module = importlib.import_module("models.DLinear")
        ACTIVE_MODEL_CLASS = module.Model
        ACTIVE_REPO = root

    elif backbone == "PatchTST":
        root = (
            REPOS[
                backbone
            ]
            / "PatchTST_supervised"
        )

        sys.path.insert(
            0,
            str(
                root
            ),
        )

        module = importlib.import_module(
            "models.PatchTST"
        )

        ACTIVE_MODEL_CLASS = (
            module.Model
        )

        ACTIVE_REPO = root

    elif backbone == "iTransformer":
        root = REPOS[
            backbone
        ]

        sys.path.insert(
            0,
            str(
                root
            ),
        )

        module = importlib.import_module(
            "model.iTransformer"
        )

        ACTIVE_MODEL_CLASS = (
            module.Model
        )

        ACTIVE_REPO = root

    elif backbone == "TimeMixer":
        root = REPOS[
            backbone
        ]

        sys.path.insert(
            0,
            str(
                root
            ),
        )

        module = importlib.import_module(
            "models.TimeMixer"
        )

        ACTIVE_MODEL_CLASS = (
            module.Model
        )

        ACTIVE_REPO = root

    else:
        raise ValueError(
            backbone
        )

    print(
        f"Activated {backbone}: "
        f"{Path(module.__file__).resolve()}"
    )

    return ACTIVE_MODEL_CLASS


In [ ]:

def set_seed(
    seed,
):
    random.seed(
        int(
            seed
        )
    )

    np.random.seed(
        int(
            seed
        )
    )

    torch.manual_seed(
        int(
            seed
        )
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            int(
                seed
            )
        )

    torch.backends.cudnn.benchmark = False


def load_torch(
    path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


def direct_config(
    backbone,
    horizon,
):
    r = DIRECT_RECIPES[
        backbone
    ]

    if backbone == "DLinear":
        return SimpleNamespace(
            task_name="long_term_forecast",
            seq_len=r["seq_len"],
            label_len=r["label_len"],
            pred_len=int(horizon),
            individual=bool(r["individual"]),
            enc_in=n_channels,
            moving_avg=int(r["moving_avg"]),
        )

    if backbone == "PatchTST":
        return SimpleNamespace(
            enc_in=
                n_channels,
            seq_len=
                r[
                    "seq_len"
                ],
            pred_len=
                int(
                    horizon
                ),
            e_layers=
                r[
                    "e_layers"
                ],
            n_heads=
                r[
                    "n_heads"
                ],
            d_model=
                r[
                    "d_model"
                ],
            d_ff=
                r[
                    "d_ff"
                ],
            dropout=
                r[
                    "dropout"
                ],
            fc_dropout=
                r[
                    "fc_dropout"
                ],
            head_dropout=
                r[
                    "head_dropout"
                ],
            individual=
                0,
            patch_len=
                r[
                    "patch_len"
                ],
            stride=
                r[
                    "stride"
                ],
            padding_patch=
                "end",
            revin=
                1,
            affine=
                0,
            subtract_last=
                0,
            decomposition=
                0,
            kernel_size=
                25,
        )

    if backbone == "iTransformer":
        return SimpleNamespace(
            task_name=
                "long_term_forecast",
            seq_len=
                r[
                    "seq_len"
                ],
            label_len=
                r[
                    "label_len"
                ],
            pred_len=
                int(
                    horizon
                ),
            enc_in=
                n_channels,
            dec_in=
                n_channels,
            c_out=
                n_channels,
            d_model=
                r[
                    "d_model"
                ],
            n_heads=
                r[
                    "n_heads"
                ],
            e_layers=
                r[
                    "e_layers"
                ],
            d_layers=
                1,
            d_ff=
                r[
                    "d_ff"
                ],
            moving_avg=
                25,
            factor=
                r[
                    "factor"
                ],
            distil=
                True,
            dropout=
                r[
                    "dropout"
                ],
            embed=
                "timeF",
            freq=
                "d",
            activation=
                "gelu",
            output_attention=
                False,
            use_norm=
                1,
            class_strategy=
                "projection",
        )

    if backbone == "TimeMixer":
        return SimpleNamespace(
            task_name=
                "long_term_forecast",
            seq_len=
                r[
                    "seq_len"
                ],
            label_len=
                r[
                    "label_len"
                ],
            pred_len=
                int(
                    horizon
                ),
            top_k=
                5,
            num_kernels=
                6,
            enc_in=
                n_channels,
            dec_in=
                n_channels,
            c_out=
                n_channels,
            d_model=
                r[
                    "d_model"
                ],
            n_heads=
                r[
                    "n_heads"
                ],
            e_layers=
                r[
                    "e_layers"
                ],
            d_layers=
                1,
            d_ff=
                r[
                    "d_ff"
                ],
            moving_avg=
                25,
            factor=
                r[
                    "factor"
                ],
            distil=
                True,
            dropout=
                r[
                    "dropout"
                ],
            embed=
                "timeF",
            freq=
                "d",
            activation=
                "gelu",
            output_attention=
                False,
            channel_independence=
                1,
            decomp_method=
                "moving_avg",
            use_norm=
                1,
            down_sampling_layers=
                r[
                    "down_sampling_layers"
                ],
            down_sampling_window=
                r[
                    "down_sampling_window"
                ],
            down_sampling_method=
                r[
                    "down_sampling_method"
                ],
            use_future_temporal_feature=
                0,
            features=
                "M",
        )

    raise ValueError(
        backbone
    )


def build_direct_model(
    backbone,
    horizon,
):
    if ACTIVE_MODEL_CLASS is None:
        raise RuntimeError(
            "Call activate_backbone() first."
        )

    cfg = direct_config(
        backbone,
        horizon,
    )

    model = ACTIVE_MODEL_CLASS(
        cfg
    ).float().to(
        DEVICE
    )

    return (
        model,
        cfg,
    )


def direct_seq_len(
    backbone,
):
    return int(
        DIRECT_RECIPES[
            backbone
        ][
            "seq_len"
        ]
    )


def make_direct_batch(
    z,
    anchors,
    backbone,
    horizon,
):
    anchors = np.asarray(
        anchors,
        dtype=np.int64,
    )

    L = direct_seq_len(
        backbone
    )

    x_idx = (
        anchors[
            :,
            None
        ]
        - L
        + np.arange(
            L
        )[
            None,
            :
        ]
    )

    y_idx = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    x = z[
        x_idx,
        :
    ].astype(
        np.float32
    )

    y = z[
        y_idx,
        :
    ].astype(
        np.float32
    )

    return (
        x,
        y,
    )


def forward_direct(
    backbone,
    model,
    x,
    y,
    horizon,
):
    x_t = torch.from_numpy(
        x
    ).to(
        DEVICE
    )

    y_t = torch.from_numpy(
        y
    ).to(
        DEVICE
    )

    if backbone == "DLinear":
        pred = model(
            x_t,
            None,
            None,
            None,
        )[
            :,
            -horizon:,
            :
        ]

    elif backbone == "PatchTST":
        pred = model(
            x_t
        )[
            :,
            -horizon:,
            :
        ]

    elif backbone == "iTransformer":
        label_len = DIRECT_RECIPES[
            backbone
        ][
            "label_len"
        ]

        dec_inp = torch.cat(
            [
                x_t[
                    :,
                    -label_len:,
                    :
                ],
                torch.zeros_like(
                    y_t
                ),
            ],
            dim=1,
        )

        pred = model(
            x_t,
            None,
            dec_inp,
            None,
        )[
            :,
            -horizon:,
            :
        ]

    elif backbone == "TimeMixer":
        pred = model(
            x_t,
            None,
            None,
            None,
        )[
            :,
            -horizon:,
            :
        ]

    else:
        raise ValueError(
            backbone
        )

    return (
        pred.float(),
        y_t.float(),
        x_t,
    )


@torch.no_grad()
def direct_residual_block(
    model,
    z,
    marks_unused,
    anchors,
    horizon,
):
    backbone = CURRENT_BACKBONE

    x, y = make_direct_batch(
        z,
        anchors,
        backbone,
        horizon,
    )

    pred, true, x_t = forward_direct(
        backbone,
        model,
        x,
        y,
        horizon,
    )

    current = x_t[
        :,
        -1:,
        :
    ].float()

    return (
        pred
        - current,
        true
        - current,
    )


In [ ]:

def backbone_dirs(
    backbone,
):
    root = (
        ROOT
        / backbone
    )

    dirs = {
        "root":
            root,
        "full_direct":
            root
            / "full_direct",
        "fold_direct":
            root
            / "fold_direct",
        "oof":
            root
            / "oof",
        "validation":
            root
            / "validation",
        "memory_emb":
            SHARED_MEMORY_EMB_DIR,
        "gate":
            root
            / "gate",
        "history":
            root
            / "history",
        "paired":
            root
            / "paired_test",
        "channel":
            root
            / "channel_test",
        "calibration":
            root
            / "calibration",
    }

    for p in dirs.values():
        if isinstance(
            p,
            Path,
        ):
            p.mkdir(
                parents=True,
                exist_ok=True,
            )

    return dirs


def full_direct_path(
    backbone,
    horizon,
):
    return (
        backbone_dirs(
            backbone
        )[
            "full_direct"
        ]
        / (
            f"Solar_{backbone}_"
            f"H{horizon}.pt"
        )
    )


def full_direct_history_path(
    backbone,
    horizon,
):
    return (
        backbone_dirs(
            backbone
        )[
            "history"
        ]
        / (
            f"Solar_{backbone}_"
            f"H{horizon}_full.csv"
        )
    )


def train_anchors_for_direct(
    backbone,
    boundary,
    horizon,
):
    L = direct_seq_len(
        backbone
    )

    return np.arange(
        L,
        int(
            boundary
        )
        - int(
            horizon
        )
        + 1,
        dtype=np.int64,
    )


@torch.no_grad()
def evaluate_direct_anchors(
    backbone,
    model,
    z,
    anchors,
    horizon,
    batch_size,
):
    model.eval()

    sse = 0.0
    sae = 0.0
    n = 0

    batch_mse = []

    for i in range(
        0,
        len(
            anchors
        ),
        batch_size,
    ):
        a = anchors[
            i:
            i+batch_size
        ]

        x, y = make_direct_batch(
            z,
            a,
            backbone,
            horizon,
        )

        pred, true, _ = forward_direct(
            backbone,
            model,
            x,
            y,
            horizon,
        )

        e = (
            pred
            - true
        )

        batch_mse.append(
            float(
                (
                    e
                    * e
                ).mean()
            )
        )

        sse += float(
            (
                e
                * e
            ).sum()
        )

        sae += float(
            e.abs().sum()
        )

        n += e.numel()

    return {
        "MSE":
            sse
            / n,
        "MAE":
            sae
            / n,
        "BatchAverageMSE":
            float(
                np.mean(
                    batch_mse
                )
            ),
        "Windows":
            len(
                anchors
            ),
    }


def set_epoch_lr_type1(
    optimizer,
    base_lr,
    epoch,
):
    lr = (
        base_lr
        * (
            0.5
            ** max(
                0,
                epoch
                - 1,
            )
        )
    )

    for group in optimizer.param_groups:
        group[
            "lr"
        ] = lr

    return lr


def train_or_load_full_direct(
    backbone,
    horizon,
):
    path = full_direct_path(
        backbone,
        horizon,
    )

    r = DIRECT_RECIPES[
        backbone
    ]

    model, cfg = build_direct_model(
        backbone,
        horizon,
    )

    if (
        RESUME
        and path.is_file()
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            f"Loaded full direct | "
            f"{backbone} H={horizon} | "
            f"best={ckpt['BestValMSE']:.6f}"
            f"@{ckpt['BestEpoch']}"
        )

        return (
            model,
            ckpt,
        )

    set_seed(
        r[
            "seed"
        ]
    )

    train_anchors = (
        train_anchors_for_direct(
            backbone,
            train_end,
            horizon,
        )
    )

    val_anchors = eval_anchors(
        train_end,
        val_end,
        horizon,
        stride=1,
        lookback=
            direct_seq_len(
                backbone
            ),
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=
            r[
                "learning_rate"
            ],
        weight_decay=
            r[
                "weight_decay"
            ],
    )

    scheduler = None

    if r[
        "scheduler"
    ] in {
        "TST",
        "OneCycle",
    }:
        steps_per_epoch = max(
            1,
            int(
                np.ceil(
                    len(
                        train_anchors
                    )
                    / r[
                        "batch_size"
                    ]
                )
            ),
        )

        scheduler = (
            torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                steps_per_epoch=
                    steps_per_epoch,
                pct_start=
                    r[
                        "pct_start"
                    ],
                epochs=
                    r[
                        "train_epochs"
                    ],
                max_lr=
                    r[
                        "learning_rate"
                    ],
            )
        )

    best_val = float(
        "inf"
    )

    best_epoch = -1
    best_state = None
    wait = 0
    history = []

    rng = np.random.default_rng(
        r[
            "seed"
        ]
        + horizon
    )

    for epoch in range(
        1,
        r[
            "train_epochs"
        ]
        + 1,
    ):
        model.train()

        if r[
            "scheduler"
        ] == "type1":
            current_lr = set_epoch_lr_type1(
                optimizer,
                r[
                    "learning_rate"
                ],
                epoch,
            )

        order = rng.permutation(
            train_anchors
        )

        losses = []
        t0 = time.time()

        for left in range(
            0,
            len(
                order
            ),
            r[
                "batch_size"
            ],
        ):
            a = order[
                left:
                left
                + r[
                    "batch_size"
                ]
            ]

            if len(
                a
            ) == 0:
                continue

            x, y = make_direct_batch(
                z_full,
                a,
                backbone,
                horizon,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            pred, true, _ = forward_direct(
                backbone,
                model,
                x,
                y,
                horizon,
            )

            loss = F.mse_loss(
                pred,
                true,
            )

            loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            losses.append(
                float(
                    loss.item()
                )
            )

        val = evaluate_direct_anchors(
            backbone,
            model,
            z_full,
            val_anchors,
            horizon,
            r[
                "eval_batch"
            ],
        )

        # Validation-only global MSE checkpoint selection.
        val_mse = val[
            "MSE"
        ]

        if (
            best_state is None
            or val_mse
            < best_val
            - 1e-12
        ):
            best_val = val_mse
            best_epoch = epoch

            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()
                for k, v
                in model.state_dict().items()
            }

            wait = 0
        else:
            wait += 1

        if scheduler is not None:
            current_lr = optimizer.param_groups[
                0
            ][
                "lr"
            ]

        history.append({
            "Epoch":
                epoch,
            "TrainMSE":
                float(
                    np.mean(
                        losses
                    )
                ),
            "ValMSE":
                val_mse,
            "ValMAE":
                val[
                    "MAE"
                ],
            "BestValMSE":
                best_val,
            "BestEpoch":
                best_epoch,
            "LR":
                current_lr,
            "Seconds":
                time.time()
                - t0,
        })

        pd.DataFrame(
            history
        ).to_csv(
            full_direct_history_path(
                backbone,
                horizon,
            ),
            index=False,
        )

        print(
            f"{backbone:12s} "
            f"H={horizon:3d} "
            f"ep={epoch:03d} | "
            f"train={history[-1]['TrainMSE']:.6f} | "
            f"val={val_mse:.6f} | "
            f"best={best_val:.6f}@{best_epoch} | "
            f"wait={wait}/{r['patience']}"
        )

        if (
            wait
            >= r[
                "patience"
            ]
        ):
            break

    if best_state is None:
        raise RuntimeError(
            f"No full direct checkpoint for "
            f"{backbone} H={horizon}."
        )

    model.load_state_dict(
        best_state
    )

    model.eval()

    ckpt = {
        "Backbone":
            backbone,
        "Dataset":
            "Solar",
        "Horizon":
            int(
                horizon
            ),
        "SeqLen":
            direct_seq_len(
                backbone
            ),
        "BestEpoch":
            int(
                best_epoch
            ),
        "BestValMSE":
            float(
                best_val
            ),
        "Recipe":
            r,
        "StateDict":
            best_state,
    }

    torch.save(
        ckpt,
        path,
    )

    return (
        model,
        ckpt,
    )


In [ ]:

direct_baseline_rows = []

for backbone in BACKBONES:
    CURRENT_BACKBONE = (
        backbone
    )

    DIRS = backbone_dirs(
        backbone
    )

    activate_backbone(
        backbone
    )

    for horizon in HORIZONS:
        model, ckpt = (
            train_or_load_full_direct(
                backbone,
                horizon,
            )
        )

        test_anchors = eval_anchors(
            val_end,
            test_end,
            horizon,
            stride=1,
            lookback=
                direct_seq_len(
                    backbone
                ),
        )

        direct_test = (
            evaluate_direct_anchors(
                backbone,
                model,
                z_full,
                test_anchors,
                horizon,
                DIRECT_RECIPES[
                    backbone
                ][
                    "eval_batch"
                ],
            )
        )

        direct_baseline_rows.append({
            "Backbone":
                backbone,
            "Dataset":
                "Solar",
            "Horizon":
                horizon,
            "SeqLen":
                direct_seq_len(
                    backbone
                ),
            "BestEpoch":
                ckpt[
                    "BestEpoch"
                ],
            "BestValMSE":
                ckpt[
                    "BestValMSE"
                ],
            "DirectTestMSE":
                direct_test[
                    "MSE"
                ],
            "DirectTestMAE":
                direct_test[
                    "MAE"
                ],
            "TestWindows":
                direct_test[
                    "Windows"
                ],
        })

        print(
            f"DIRECT | {backbone:12s} "
            f"H={horizon:3d} | "
            f"MSE={direct_test['MSE']:.6f} | "
            f"MAE={direct_test['MAE']:.6f}"
        )

        del (
            model,
            ckpt,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


direct_baseline_df = pd.DataFrame(
    direct_baseline_rows
)

direct_baseline_df.to_csv(
    ROOT
    / "direct_baselines.csv",
    index=False,
)

display(
    direct_baseline_df
)


In [ ]:
def inv_softplus(x):
    return math.log(math.exp(float(x)) - 1.0)


class PredictivePatchEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_proj = nn.Linear(REP_PATCH_LEN, REP_D_MODEL)
        self.pos_embed = nn.Parameter(torch.zeros(1, REP_NUM_PATCHES, REP_D_MODEL))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=REP_D_MODEL, nhead=REP_N_HEADS,
            dim_feedforward=REP_D_FF, dropout=REP_DROPOUT,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=REP_LAYERS)
        self.norm = nn.LayerNorm(REP_D_MODEL)
        self.proj = nn.Linear(REP_D_MODEL, REP_DIM)

    def forward(self, x):
        p = x.unfold(1, REP_PATCH_LEN, REP_PATCH_STRIDE)
        h = self.patch_proj(p) + self.pos_embed[:, :p.shape[1]]
        h = self.encoder(h).mean(dim=1)
        h = self.proj(self.norm(h))
        return F.normalize(h, dim=-1, eps=1e-8)


class EmbeddingOnlyRetriever(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = PredictivePatchEncoder()
        self.raw_gamma = nn.Parameter(
            torch.tensor(inv_softplus(1.0), dtype=torch.float32)
        )

    @property
    def gamma(self):
        return F.softplus(self.raw_gamma)

    def encode(self, x):
        return self.encoder(x)


def ret_amp():
    if RETRIEVER_USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


STRONG_FORECASTER_ROOT = Path("/data/dataset/strong_forecaster")
SOLAR_RETRIEVER_ROOT = STRONG_FORECASTER_ROOT / "multidataset_crossfit_screening"
SOLAR_FULL_RETRIEVER_DIR = STRONG_FORECASTER_ROOT / "solar_predictive_representation" / "checkpoints"
SOLAR_FOLD_RETRIEVER_DIR = STRONG_FORECASTER_ROOT / "solar_crossfit_adaptive_gate" / "fold_retriever_checkpoints"


def _discover_solar_retriever(horizon, fold=None):
    horizon = int(horizon)
    if fold is None:
        preferred = SOLAR_FULL_RETRIEVER_DIR / f"Solar_L96_H{horizon}_EmbeddingOnly_Listwise_seed0.pt"
        patterns = [f"Solar_L96_H{horizon}_EmbeddingOnly_Listwise_seed0.pt", f"Solar_H{horizon}_seed0.pt", f"*Solar*H{horizon}*seed0*.pt"]
    else:
        fold = int(fold)
        preferred = SOLAR_FOLD_RETRIEVER_DIR / f"Solar_H{horizon}_Fold{fold}_EmbeddingOnly.pt"
        patterns = [f"Solar_H{horizon}_Fold{fold}_EmbeddingOnly.pt", f"Solar_H{horizon}_F{fold}_seed0.pt", f"*Solar*H{horizon}*Fold{fold}*.pt", f"*Solar*H{horizon}*F{fold}*seed0*.pt"]
    if preferred.is_file():
        return preferred
    hits=[]
    for pattern in patterns:
        for p in STRONG_FORECASTER_ROOT.rglob(pattern):
            low=str(p).lower()
            if "direct" in low or "/gate/" in low or "_gate" in low or ROOT in p.parents:
                continue
            hits.append(Path(p))
    if not hits:
        raise FileNotFoundError(f"No frozen Solar retriever found for H={horizon}, fold={fold}")
    return sorted(set(hits), key=lambda p:(0 if "retriever" in str(p).lower() else 1, len(p.parts), str(p)))[0]


def full_retriever_ckpt_path(name, horizon):
    if name != "Solar": raise ValueError(name)
    return _discover_solar_retriever(horizon, fold=None)


def fold_retriever_ckpt_path(name, horizon, fold):
    if name != "Solar": raise ValueError(name)
    return _discover_solar_retriever(horizon, fold=fold)


def load_frozen_retriever(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    ckpt=load_torch(path)
    state=ckpt
    if isinstance(ckpt,dict):
        for key in ["StateDict","state_dict","ModelState","model_state_dict","RetrieverState","retriever_state","RetrieverStateDict","retriever_state_dict","Model"]:
            if isinstance(ckpt.get(key),dict):
                state=ckpt[key]; break
    state=dict(state)
    remap={"encoder.out_norm.weight":"encoder.norm.weight","encoder.out_norm.bias":"encoder.norm.bias","encoder.out_proj.weight":"encoder.proj.weight","encoder.out_proj.bias":"encoder.proj.bias"}
    for old,new in remap.items():
        if old in state and new not in state: state[new]=state.pop(old)
    for prefix in ["module.","model.","retriever."]:
        keys=list(state)
        if keys and all(k.startswith(prefix) for k in keys):
            state={k[len(prefix):]:v for k,v in state.items()}
    model=EmbeddingOnlyRetriever().to(DEVICE)
    model.load_state_dict(state,strict=True)
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    print("Loaded frozen retriever:",path)
    return model,ckpt


In [ ]:

# -----------------------------------------------------------------------------
# Auto-training for missing Solar embedding-only retrievers.
# -----------------------------------------------------------------------------

def _ret_pattern_np(x):
    x = np.asarray(x, dtype=np.float32)
    xc = x - x.mean(axis=-1, keepdims=True)
    norm = np.linalg.norm(xc, axis=-1, keepdims=True)
    return np.where(norm > EPS, xc / np.maximum(norm, EPS), 0.0).astype(np.float32)


def _ret_extract_channel(z, channel, anchors, horizon):
    anchors = np.asarray(anchors, dtype=np.int64)
    pi = anchors[:, None] - RET_SEQ_LEN + np.arange(RET_SEQ_LEN)[None, :]
    fi = anchors[:, None] + np.arange(horizon)[None, :]
    past = z[pi, channel].astype(np.float32)
    future = z[fi, channel].astype(np.float32)
    current = z[anchors - 1, channel].astype(np.float32)
    future_residual = (future - current[:, None]).astype(np.float32)
    return past, future_residual


def _ret_soft_targets(dist):
    dist = np.asarray(dist, dtype=np.float32)
    mu = dist.mean(axis=1, keepdims=True)
    sd = dist.std(axis=1, keepdims=True)
    sd = np.maximum(sd, 1e-6)
    zdist = (dist - mu) / sd
    logits = -zdist / RETRIEVER_TAU
    logits = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(logits).astype(np.float32)
    p /= np.maximum(p.sum(axis=1, keepdims=True), 1e-12)
    return p.astype(np.float32)


def _prepare_retriever_problem(z, prefix, horizon, query_stride=RETRIEVER_QUERY_STRIDE):
    """Build leakage-free same-channel Pattern Top-M listwise supervision."""
    prefix = int(prefix)
    horizon = int(horizon)
    C = z.shape[1]

    memory_end = int(RETRIEVER_MEMORY_FRACTION * prefix)
    memory_end = max(memory_end, RET_SEQ_LEN + horizon + MEMORY_STRIDE)

    memory_anchors = np.arange(
        RET_SEQ_LEN,
        memory_end - horizon + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )
    if len(memory_anchors) < TOP_K:
        raise ValueError(
            f"Too few memory candidates: prefix={prefix}, H={horizon}, M={len(memory_anchors)}"
        )

    m_use = min(RETRIEVER_CANDIDATE_M, len(memory_anchors))

    # Query origins are strictly after the fixed internal memory region.
    q_last = prefix - horizon
    if q_last < memory_end:
        raise ValueError(
            f"No retriever supervision interval: prefix={prefix}, H={horizon}, memory_end={memory_end}"
        )

    query_anchors = np.arange(
        memory_end,
        q_last + 1,
        query_stride,
        dtype=np.int64,
    )
    if len(query_anchors) < 4:
        raise ValueError(
            f"Too few retriever queries: prefix={prefix}, H={horizon}, n={len(query_anchors)}"
        )

    memory_past = np.empty((C, len(memory_anchors), RET_SEQ_LEN), dtype=np.float32)
    memory_pattern = np.empty_like(memory_past)
    memory_future = np.empty((C, len(memory_anchors), horizon), dtype=np.float32)

    for c in range(C):
        p, f = _ret_extract_channel(z, c, memory_anchors, horizon)
        memory_past[c] = p
        memory_pattern[c] = _ret_pattern_np(p)
        memory_future[c] = f

    # We flatten anchor x channel pairs.
    n_pairs = len(query_anchors) * C
    query_past = np.empty((n_pairs, RET_SEQ_LEN), dtype=np.float32)
    candidate_idx = np.empty((n_pairs, m_use), dtype=np.int32)
    candidate_channel = np.empty(n_pairs, dtype=np.int16)
    future_dist = np.empty((n_pairs, m_use), dtype=np.float32)
    pair_anchor = np.empty(n_pairs, dtype=np.int64)

    row = 0
    for c in range(C):
        qpast, qfuture = _ret_extract_channel(z, c, query_anchors, horizon)
        qpat = _ret_pattern_np(qpast)
        sim = qpat @ memory_pattern[c].T

        # Pattern Top-M: argpartition then exact descending order within the pool.
        if m_use < sim.shape[1]:
            idx0 = np.argpartition(sim, -m_use, axis=1)[:, -m_use:]
            score0 = np.take_along_axis(sim, idx0, axis=1)
            order = np.argsort(-score0, axis=1)
            idx = np.take_along_axis(idx0, order, axis=1)
        else:
            idx = np.argsort(-sim, axis=1)[:, :m_use]

        cand_future = memory_future[c][idx]
        dist = ((cand_future - qfuture[:, None, :]) ** 2).mean(axis=2).astype(np.float32)

        sl = slice(row, row + len(query_anchors))
        query_past[sl] = qpast
        candidate_idx[sl] = idx.astype(np.int32)
        candidate_channel[sl] = c
        future_dist[sl] = dist
        pair_anchor[sl] = query_anchors
        row += len(query_anchors)

    target_prob = _ret_soft_targets(future_dist)

    # Chronological Phase-A split by query origin, not by randomly shuffled rows.
    unique_a = np.unique(pair_anchor)
    cut_n = max(1, min(len(unique_a) - 1, int(RETRIEVER_PHASEA_TRAIN_FRACTION * len(unique_a))))
    cut_anchor = unique_a[cut_n]
    train_mask = pair_anchor < cut_anchor
    val_mask = pair_anchor >= cut_anchor

    if train_mask.sum() == 0 or val_mask.sum() == 0:
        raise ValueError(
            f"Invalid retriever internal split: prefix={prefix}, H={horizon}, "
            f"train={train_mask.sum()}, val={val_mask.sum()}"
        )

    return {
        "prefix": prefix,
        "memory_end": memory_end,
        "memory_anchors": memory_anchors,
        "memory_past": memory_past,
        "query_past": query_past,
        "candidate_idx": candidate_idx,
        "candidate_channel": candidate_channel,
        "future_dist": future_dist,
        "target_prob": target_prob,
        "pair_anchor": pair_anchor,
        "train_ids": np.where(train_mask)[0].astype(np.int64),
        "val_ids": np.where(val_mask)[0].astype(np.int64),
        "all_ids": np.arange(n_pairs, dtype=np.int64),
        "candidate_m": m_use,
    }


def _retriever_batch(model, problem, ids):
    ids = np.asarray(ids, dtype=np.int64)
    q_np = problem["query_past"][ids]
    ch = problem["candidate_channel"][ids].astype(np.int64)
    ci = problem["candidate_idx"][ids].astype(np.int64)
    cand_np = problem["memory_past"][ch[:, None], ci]
    tgt_np = problem["target_prob"][ids]

    q = torch.from_numpy(q_np).to(DEVICE)
    cand = torch.from_numpy(cand_np.reshape(-1, RET_SEQ_LEN)).to(DEVICE)
    target = torch.from_numpy(tgt_np).to(DEVICE)

    qemb = model.encode(q)
    cemb = model.encode(cand).reshape(len(ids), problem["candidate_m"], REP_DIM)
    score = model.gamma * torch.einsum("bd,bmd->bm", qemb, cemb)
    loss = -(target * F.log_softmax(score, dim=1)).sum(dim=1).mean()
    return loss, score


@torch.no_grad()
def _retriever_val_analog_mse(model, problem, ids):
    model.eval()
    total = 0.0
    count = 0
    for i in range(0, len(ids), RETRIEVER_BATCH_QUERIES):
        bid = ids[i:i + RETRIEVER_BATCH_QUERIES]
        q_np = problem["query_past"][bid]
        ch = problem["candidate_channel"][bid].astype(np.int64)
        ci = problem["candidate_idx"][bid].astype(np.int64)
        cand_np = problem["memory_past"][ch[:, None], ci]

        q = torch.from_numpy(q_np).to(DEVICE)
        cand = torch.from_numpy(cand_np.reshape(-1, RET_SEQ_LEN)).to(DEVICE)
        with ret_amp():
            qemb = model.encode(q)
            cemb = model.encode(cand).reshape(len(bid), problem["candidate_m"], REP_DIM)
        score = model.gamma.float() * torch.einsum("bd,bmd->bm", qemb.float(), cemb.float())
        k = min(TOP_K, score.shape[1])
        top = torch.topk(score, k, dim=1).indices.cpu().numpy()
        d = problem["future_dist"][bid]
        selected = np.take_along_axis(d, top, axis=1)
        total += float(selected.sum())
        count += selected.size
        del q, cand, qemb, cemb, score
    return total / max(count, 1)


def _fit_retriever_epochs(model, problem, ids, epochs, seed, verbose_prefix):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=RETRIEVER_LR,
        weight_decay=RETRIEVER_WD,
    )
    rng = np.random.default_rng(seed + 12345)
    hist = []
    for epoch in range(1, epochs + 1):
        model.train()
        order = np.asarray(ids, dtype=np.int64).copy()
        rng.shuffle(order)
        losses = []
        for i in range(0, len(order), RETRIEVER_BATCH_QUERIES):
            bid = order[i:i + RETRIEVER_BATCH_QUERIES]
            optimizer.zero_grad(set_to_none=True)
            loss, _ = _retriever_batch(model, problem, bid)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), RETRIEVER_GRAD_CLIP)
            optimizer.step()
            losses.append(float(loss.item()))
        mean_loss = float(np.mean(losses)) if losses else float("nan")
        hist.append(mean_loss)
        print(
            f"{verbose_prefix} ep={epoch:02d}/{epochs:02d} | "
            f"loss={mean_loss:.6f} | gamma={float(model.gamma.detach().cpu()):.4f}"
        )
    return hist


def train_or_load_solar_retriever(path, z, prefix, horizon, tag):
    path = Path(path)
    if RESUME and path.is_file() and not FORCE:
        print("Loaded frozen retriever:", path)
        return path

    set_seed(RETRIEVER_SEED)
    problem = _prepare_retriever_problem(z, prefix, horizon)

    print(
        f"Retriever prep | {tag} | H={horizon} | prefix={prefix} | "
        f"memory={len(problem['memory_anchors'])}/ch | candidates={problem['candidate_m']} | "
        f"train_pairs={len(problem['train_ids'])} | val_pairs={len(problem['val_ids'])}"
    )

    # Phase A: chronological internal validation chooses epoch count only.
    model = EmbeddingOnlyRetriever().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=RETRIEVER_LR,
        weight_decay=RETRIEVER_WD,
    )
    rng = np.random.default_rng(RETRIEVER_SEED + horizon + prefix)

    best_val = float("inf")
    best_epoch = 1
    wait = 0
    phase_a_history = []

    for epoch in range(1, RETRIEVER_MAX_EPOCHS + 1):
        model.train()
        order = problem["train_ids"].copy()
        rng.shuffle(order)
        losses = []

        for i in range(0, len(order), RETRIEVER_BATCH_QUERIES):
            bid = order[i:i + RETRIEVER_BATCH_QUERIES]
            optimizer.zero_grad(set_to_none=True)
            loss, _ = _retriever_batch(model, problem, bid)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), RETRIEVER_GRAD_CLIP)
            optimizer.step()
            losses.append(float(loss.item()))

        val_mse = _retriever_val_analog_mse(model, problem, problem["val_ids"])
        train_loss = float(np.mean(losses)) if losses else float("nan")
        phase_a_history.append((epoch, train_loss, val_mse, float(model.gamma.detach().cpu())))

        print(
            f"Retriever {tag:>8s} H={horizon:>3d} ep={epoch:02d} | "
            f"loss={train_loss:.6f} | valAnalog={val_mse:.6f} | "
            f"gamma={float(model.gamma.detach().cpu()):.4f}"
        )

        if val_mse < best_val - 1e-10:
            best_val = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1
            if wait >= RETRIEVER_PATIENCE:
                break

    del model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Phase B: reinitialize and refit on every historical query available before prefix.
    set_seed(RETRIEVER_SEED)
    model = EmbeddingOnlyRetriever().to(DEVICE)
    phase_b_loss = _fit_retriever_epochs(
        model,
        problem,
        problem["all_ids"],
        best_epoch,
        RETRIEVER_SEED + horizon + prefix,
        f"Refit {tag} H={horizon}",
    )

    path.parent.mkdir(parents=True, exist_ok=True)
    ckpt = {
        "StateDict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "Dataset": "Solar",
        "Horizon": int(horizon),
        "Prefix": int(prefix),
        "Tag": str(tag),
        "BestEpoch": int(best_epoch),
        "BestValAnalogMSE": float(best_val),
        "Gamma": float(model.gamma.detach().cpu()),
        "Recipe": {
            "retrieval_seq_len": RET_SEQ_LEN,
            "memory_stride": MEMORY_STRIDE,
            "candidate_m": int(problem["candidate_m"]),
            "top_k": TOP_K,
            "target_temperature": RETRIEVER_TAU,
            "optimizer": "AdamW",
            "learning_rate": RETRIEVER_LR,
            "weight_decay": RETRIEVER_WD,
            "batch_queries": RETRIEVER_BATCH_QUERIES,
            "max_epochs": RETRIEVER_MAX_EPOCHS,
            "patience": RETRIEVER_PATIENCE,
            "gradient_clip": RETRIEVER_GRAD_CLIP,
            "query_stride": RETRIEVER_QUERY_STRIDE,
            "internal_memory_fraction": RETRIEVER_MEMORY_FRACTION,
            "phase_a_train_fraction": RETRIEVER_PHASEA_TRAIN_FRACTION,
            "seed": RETRIEVER_SEED,
            "candidate_rule": "same-channel Pattern Top-M",
            "target_rule": "query-wise standardized future MSE softmax",
        },
        "PhaseAHistory": phase_a_history,
        "PhaseBLoss": phase_b_loss,
    }
    torch.save(ckpt, path)
    print(
        f"Saved frozen retriever | {tag} H={horizon} | "
        f"bestEpoch={best_epoch} | valAnalog={best_val:.6f} | {path}"
    )

    del model, problem
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return path


def ensure_solar_retriever_checkpoints():
    # Full retrievers use train-only normalization and train-only supervision.
    for horizon in HORIZONS:
        train_or_load_solar_retriever(
            full_retriever_ckpt_path("Solar", horizon),
            DATA["Solar"]["z"],
            DATA["Solar"]["train_end"],
            horizon,
            "Full",
        )

        # Fold retrievers are prefix-only; each is trained before its OOF interval begins.
        for fold, (p0, _p1) in enumerate(FOLDS, start=1):
            prefix = int(p0 * DATA["Solar"]["train_end"])
            z_prefix, _ = prefix_normalize(DATA["Solar"]["raw"], prefix)
            train_or_load_solar_retriever(
                fold_retriever_ckpt_path("Solar", horizon, fold),
                z_prefix,
                prefix,
                horizon,
                f"Fold{fold}",
            )
            del z_prefix
            gc.collect()


ensure_solar_retriever_checkpoints()

# Final preflight after auto-training.
retriever_preflight_rows = []
for horizon in HORIZONS:
    full_path = full_retriever_ckpt_path("Solar", horizon)
    retriever_preflight_rows.append({
        "Horizon": horizon,
        "Kind": "Full",
        "Path": str(full_path),
        "Exists": full_path.is_file(),
    })
    for fold in range(1, 4):
        fold_path = fold_retriever_ckpt_path("Solar", horizon, fold)
        retriever_preflight_rows.append({
            "Horizon": horizon,
            "Kind": f"Fold{fold}",
            "Path": str(fold_path),
            "Exists": fold_path.is_file(),
        })

retriever_preflight = pd.DataFrame(retriever_preflight_rows)
display(retriever_preflight)
missing = retriever_preflight[~retriever_preflight["Exists"]]
if len(missing):
    display(missing)
    raise FileNotFoundError("Solar retriever auto-training did not produce all required checkpoints.")
print("PASS: all 4 full + 12 fold Solar retriever checkpoints exist.")


In [ ]:

def retrieval_anchor_batch(
    name,
):
    C = DATA[
        name
    ][
        "n_channels"
    ]

    return max(
        1,
        TARGET_RETRIEVAL_PAIRS
        // C,
    )


def batch_pattern(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    xc = (
        x
        - x.mean(
            axis=-1,
            keepdims=True,
        )
    )

    n = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        n > EPS,
        xc
        / np.maximum(
            n,
            EPS,
        ),
        0.0,
    ).astype(
        np.float32
    )


def context7(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    short = max(
        8,
        RET_SEQ_LEN
        // 4,
    )

    m = x.mean(
        axis=-1
    )

    s = (
        x.std(
            axis=-1
        )
        + EPS
    )

    f1 = (
        x[
            ...,
            -1
        ]
        - m
    ) / s

    f2 = (
        x[
            ...,
            -short:
        ].mean(
            axis=-1
        )
        - m
    ) / s

    f3 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            -short
        ]
    ) / s

    f4 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            0
        ]
    ) / s

    df = np.diff(
        x,
        axis=-1,
    )

    ds = np.diff(
        x[
            ...,
            -short:
        ],
        axis=-1,
    )

    f5 = (
        ds.std(
            axis=-1
        )
        + EPS
    ) / (
        df.std(
            axis=-1
        )
        + EPS
    )

    t = np.linspace(
        -1.0,
        1.0,
        RET_SEQ_LEN,
        dtype=np.float32,
    )

    t = (
        t
        - t.mean()
    )

    f6 = (
        np.sum(
            t
            * (
                x
                - m[
                    ...,
                    None
                ]
            ),
            axis=-1,
        )
        / (
            np.sum(
                t
                * t
            )
            + EPS
        )
    ) / s

    a = x[
        ...,
        :-1
    ]

    b = x[
        ...,
        1:
    ]

    a = (
        a
        - a.mean(
            axis=-1,
            keepdims=True,
        )
    )

    b = (
        b
        - b.mean(
            axis=-1,
            keepdims=True,
        )
    )

    f7 = np.sum(
        a
        * b,
        axis=-1,
    ) / (
        np.sqrt(
            np.sum(
                a
                * a,
                axis=-1,
            )
            * np.sum(
                b
                * b,
                axis=-1,
            )
        )
        + EPS
    )

    return np.stack(
        [
            f1,
            f2,
            f3,
            f4,
            f5,
            f6,
            f7,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


def extract_channel(
    z,
    c,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        c,
    ].astype(
        np.float32
    )

    future = z[
        fi,
        c,
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        c,
    ].astype(
        np.float32
    )

    future_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        future_residual,
    )


def build_memory(
    z,
    channels,
    boundary,
    horizon,
):
    memory_anchors = np.arange(
        RET_SEQ_LEN,
        int(
            boundary
        )
        - horizon
        + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(
        memory_anchors
    ) < TOP_K:
        raise ValueError(
            "Insufficient admissible memory."
        )

    past = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    pattern = np.empty_like(
        past
    )

    future = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            horizon,
        ),
        dtype=np.float32,
    )

    for c in range(
        channels
    ):
        p, f = extract_channel(
            z,
            c,
            memory_anchors,
            horizon,
        )

        past[
            c
        ] = p

        pattern[
            c
        ] = batch_pattern(
            p
        )

        future[
            c
        ] = f

    return {
        "anchors":
            memory_anchors,
        "past":
            past,
        "pattern":
            pattern,
        "future":
            future,
        "M":
            len(
                memory_anchors
            ),
        "boundary":
            int(
                boundary
            ),
    }


def query_pairs(
    z,
    anchors,
    channels,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    channels = np.asarray(
        channels,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    future = z[
        fi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        channels,
    ].astype(
        np.float32
    )

    true_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        batch_pattern(
            past
        ),
        context7(
            past
        ),
        true_residual,
    )


In [ ]:

@torch.no_grad()
def encode_np(
    model,
    x,
    chunk=512,
):
    parts = []

    for i in range(
        0,
        len(
            x
        ),
        chunk,
    ):
        t = torch.from_numpy(
            x[
                i:
                i+chunk
            ]
        ).to(
            DEVICE
        )

        with ret_amp():
            e = model.encode(
                t
            ).float()

        parts.append(
            e.cpu()
        )

        del (
            t,
            e,
        )

    return torch.cat(
        parts,
        dim=0,
    ).numpy().astype(
        np.float32
    )


def memory_embedding_path(
    name,
    horizon,
    tag,
):
    return (
        SHARED_MEMORY_EMB_DIR
        / (
            f"{name}_H{horizon}_"
            f"{tag}_emb.npy"
        )
    )


@torch.no_grad()
def memory_gpu_cached(
    name,
    horizon,
    tag,
    model,
    memory,
    channels,
):
    path = memory_embedding_path(
        name,
        horizon,
        tag,
    )

    expected = (
        channels,
        memory[
            "M"
        ],
        REP_DIM,
    )

    emb_np = None

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        candidate = np.load(
            path,
            mmap_mode=None,
        )

        if (
            tuple(
                candidate.shape
            )
            == expected
        ):
            emb_np = candidate.astype(
                np.float32,
                copy=False,
            )

            print(
                "Loaded memory embedding:",
                path.name,
            )

    if emb_np is None:
        emb_np = np.empty(
            expected,
            dtype=np.float32,
        )

        print(
            "Building memory embedding:",
            path.name,
            expected,
        )

        for c in range(
            channels
        ):
            emb_np[
                c
            ] = encode_np(
                model,
                memory[
                    "past"
                ][
                    c
                ],
            )

            if (
                c == 0
                or (
                    c + 1
                )
                % 50
                == 0
                or (
                    c + 1
                    == channels
                )
            ):
                print(
                    f"  channel "
                    f"{c+1}/{channels}"
                )

        np.save(
            path,
            emb_np,
        )

    return {
        "emb":
            torch.from_numpy(
                emb_np
            ).to(
                DEVICE
            ),
        "pattern":
            torch.from_numpy(
                memory[
                    "pattern"
                ]
            ).to(
                DEVICE
            ),
        "future":
            torch.from_numpy(
                memory[
                    "future"
                ]
            ).to(
                DEVICE
            ),
    }


In [ ]:

@torch.no_grad()
def retrieve(
    model,
    memory_gpu_obj,
    z,
    anchors,
    channels,
    horizon,
):
    (
        past,
        pattern,
        ctx,
        true,
    ) = query_pairs(
        z,
        anchors,
        channels,
        horizon,
    )

    past_t = torch.from_numpy(
        past
    ).to(
        DEVICE
    )

    pattern_t = torch.from_numpy(
        pattern
    ).to(
        DEVICE
    )

    with ret_amp():
        qemb = model.encode(
            past_t
        )

    qemb = qemb.float()

    memb = memory_gpu_obj[
        "emb"
    ][
        channels
    ]

    sim = torch.bmm(
        qemb[
            :,
            None,
            :
        ],
        memb.transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    score = (
        model.gamma
        * sim
    )

    idx = torch.topk(
        score,
        TOP_K,
        dim=1,
    ).indices

    row = torch.arange(
        len(
            channels
        ),
        device=DEVICE,
    )[
        :,
        None
    ]

    pfull = torch.bmm(
        pattern_t[
            :,
            None,
            :
        ],
        memory_gpu_obj[
            "pattern"
        ][
            channels
        ].transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    return {
        "score":
            score[
                row,
                idx
            ],
        "sim":
            sim[
                row,
                idx
            ],
        "pattern":
            pfull[
                row,
                idx
            ],
        "cand":
            memory_gpu_obj[
                "future"
            ][
                channels[
                    :,
                    None
                ],
                idx,
            ],
        "ctx":
            torch.from_numpy(
                ctx
            ).to(
                DEVICE
            ),
        "true":
            torch.from_numpy(
                true
            ).to(
                DEVICE
            ),
    }


In [ ]:

class CrossFitAdaptiveGate(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                GATE_DIM,
                64,
            ),
            nn.LayerNorm(
                64
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                64,
                32,
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                32,
                1,
            ),
        )

        nn.init.normal_(
            self.net[
                -1
            ].weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            self.net[
                -1
            ].bias,
            math.log(
                0.1
                / 0.9
            ),
        )

    def forward(
        self,
        x,
    ):
        return torch.sigmoid(
            self.net(
                x
            ).squeeze(
                -1
            )
        )


def score_entropy(
    s,
):
    p = torch.softmax(
        s,
        dim=1,
    )

    return (
        -(
            p
            * torch.log(
                p.clamp_min(
                    1e-8
                )
            )
        ).sum(
            dim=1
        )
        / math.log(
            TOP_K
        )
    )


def feature_cosine(
    a,
    b,
):
    return (
        (
            a
            * b
        ).sum(
            dim=1
        )
        / (
            torch.sqrt(
                (
                    a
                    * a
                ).sum(
                    dim=1
                )
                + 1e-8
            )
            * torch.sqrt(
                (
                    b
                    * b
                ).sum(
                    dim=1
                )
                + 1e-8
            )
        )
    )


def gate_features(
    r,
    retrieval,
    direct,
):
    s = r[
        "score"
    ]

    sim = r[
        "sim"
    ]

    pattern = r[
        "pattern"
    ]

    sorted_s = torch.sort(
        s,
        dim=1,
        descending=True,
    ).values

    cand_std = r[
        "cand"
    ].std(
        dim=1,
        unbiased=False,
    )

    disp_rms = torch.sqrt(
        (
            cand_std
            * cand_std
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disp_mean = cand_std.mean(
        dim=1
    )

    direct_rms = torch.sqrt(
        (
            direct
            * direct
        ).mean(
            dim=1
        )
        + 1e-8
    )

    retrieval_rms = torch.sqrt(
        (
            retrieval
            * retrieval
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disagreement = (
        retrieval
        - direct
    )

    disagreement_rms = torch.sqrt(
        (
            disagreement
            * disagreement
        ).mean(
            dim=1
        )
        + 1e-8
    )

    relative_disagreement = (
        disagreement_rms
        / (
            direct_rms
            + retrieval_rms
            + 1e-6
        )
    )

    scalars = torch.stack(
        [
            s.mean(
                dim=1
            ),
            s.std(
                dim=1,
                unbiased=False,
            ),
            s.max(
                dim=1
            ).values,
            sorted_s[
                :,
                0
            ]
            - sorted_s[
                :,
                1
            ],
            s.max(
                dim=1
            ).values
            - s.mean(
                dim=1
            ),
            score_entropy(
                s
            ),
            sim.mean(
                dim=1
            ),
            sim.std(
                dim=1,
                unbiased=False,
            ),
            sim.max(
                dim=1
            ).values,
            pattern.mean(
                dim=1
            ),
            pattern.std(
                dim=1,
                unbiased=False,
            ),
            pattern.max(
                dim=1
            ).values,
            disp_rms,
            disp_mean,
            direct_rms,
            retrieval_rms,
            disagreement_rms,
            relative_disagreement,
            feature_cosine(
                direct,
                retrieval,
            ),
        ],
        dim=1,
    )

    out = torch.cat(
        [
            r[
                "ctx"
            ],
            scalars,
        ],
        dim=1,
    )

    if out.shape[
        1
    ] != GATE_DIM:
        raise RuntimeError(
            f"Gate feature dimension mismatch: "
            f"{out.shape}"
        )

    return out


def abc_terms(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    return torch.stack(
        [
            (
                e
                * e
            ).mean(
                dim=1
            ),
            (
                e
                * delta
            ).mean(
                dim=1
            ),
            (
                delta
                * delta
            ).mean(
                dim=1
            ),
        ],
        dim=1,
    )


In [ ]:

def fold_direct_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "fold_direct"
        ]
        / (
            f"{name}_{CURRENT_BACKBONE}_"
            f"H{horizon}_F{fold}.pt"
        )
    )


def fold_direct_history_path(
    horizon,
    fold,
):
    return (
        DIRS[
            "history"
        ]
        / (
            f"Solar_{CURRENT_BACKBONE}_"
            f"H{horizon}_F{fold}_direct.csv"
        )
    )


def train_fold_direct(
    backbone,
    horizon,
    z,
    prefix,
    fixed_epochs,
    fold,
):
    path = fold_direct_path(
        "Solar",
        horizon,
        fold,
    )

    r = DIRECT_RECIPES[
        backbone
    ]

    model, cfg = build_direct_model(
        backbone,
        horizon,
    )

    if (
        RESUME
        and path.is_file()
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded fold direct:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    seed = (
        r[
            "seed"
        ]
        + 10_000
        + horizon
        * 10
        + fold
    )

    set_seed(
        seed
    )

    anchors = train_anchors_for_direct(
        backbone,
        prefix,
        horizon,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=
            r[
                "learning_rate"
            ],
        weight_decay=
            r[
                "weight_decay"
            ],
    )

    scheduler = None

    if r[
        "scheduler"
    ] in {
        "TST",
        "OneCycle",
    }:
        steps_per_epoch = max(
            1,
            int(
                np.ceil(
                    len(
                        anchors
                    )
                    / r[
                        "batch_size"
                    ]
                )
            ),
        )

        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            steps_per_epoch=
                steps_per_epoch,
            pct_start=
                r[
                    "pct_start"
                ],
            # Preserve the full-model schedule budget.
            epochs=
                r[
                    "train_epochs"
                ],
            max_lr=
                r[
                    "learning_rate"
                ],
        )

    rng = np.random.default_rng(
        seed
        + 1
    )

    history = []

    for epoch in range(
        1,
        int(
            fixed_epochs
        )
        + 1,
    ):
        model.train()

        if r[
            "scheduler"
        ] == "type1":
            current_lr = set_epoch_lr_type1(
                optimizer,
                r[
                    "learning_rate"
                ],
                epoch,
            )

        order = rng.permutation(
            anchors
        )

        losses = []
        t0 = time.time()

        for left in range(
            0,
            len(
                order
            ),
            r[
                "batch_size"
            ],
        ):
            a = order[
                left:
                left
                + r[
                    "batch_size"
                ]
            ]

            if len(
                a
            ) == 0:
                continue

            x, y = make_direct_batch(
                z,
                a,
                backbone,
                horizon,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            pred, true, _ = forward_direct(
                backbone,
                model,
                x,
                y,
                horizon,
            )

            loss = F.mse_loss(
                pred,
                true,
            )

            loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            losses.append(
                float(
                    loss.item()
                )
            )

        if scheduler is not None:
            current_lr = optimizer.param_groups[
                0
            ][
                "lr"
            ]

        history.append({
            "Epoch":
                epoch,
            "TrainMSE":
                float(
                    np.mean(
                        losses
                    )
                ),
            "LR":
                current_lr,
            "Seconds":
                time.time()
                - t0,
        })

        pd.DataFrame(
            history
        ).to_csv(
            fold_direct_history_path(
                horizon,
                fold,
            ),
            index=False,
        )

        print(
            f"Fold {backbone:12s} "
            f"H={horizon:3d} F{fold} "
            f"ep={epoch:02d}/{fixed_epochs} | "
            f"train={history[-1]['TrainMSE']:.6f}"
        )

    model.eval()

    state = {
        k:
            v.detach()
            .cpu()
            .clone()
        for k, v
        in model.state_dict().items()
    }

    ckpt = {
        "Backbone":
            backbone,
        "Dataset":
            "Solar",
        "Horizon":
            horizon,
        "Fold":
            fold,
        "Prefix":
            int(
                prefix
            ),
        "FixedEpochs":
            int(
                fixed_epochs
            ),
        "Seed":
            seed,
        "StateDict":
            state,
    }

    torch.save(
        ckpt,
        path,
    )

    return (
        model,
        ckpt,
    )


In [ ]:

@torch.no_grad()
def collect_gate_data(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    z,
    anchors,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    direct_block = DIRECT_ANCHOR_BLOCK[
        CURRENT_BACKBONE
    ]

    features = []
    abcs = []
    anchors_out = []
    channels_out = []

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                z,
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        # [A, H, C] -> [A, C, H]
        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                z,
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            d = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            t = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            # Strong consistency check:
            # direct true residual must match retrieval true residual.
            max_true_diff = float(
                (
                    t
                    - r[
                        "true"
                    ]
                ).abs().max()
            )

            if (
                max_true_diff
                > 2e-5
            ):
                raise RuntimeError(
                    f"True residual mismatch: "
                    f"{max_true_diff}"
                )

            feat = gate_features(
                r,
                retrieval,
                d,
            )

            abc = abc_terms(
                d,
                retrieval,
                t,
            )

            features.append(
                feat.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            abcs.append(
                abc.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            anchors_out.append(
                pair_anchor
            )

            channels_out.append(
                pair_channel
            )

            del (
                r,
                retrieval,
                d,
                t,
                feat,
                abc,
            )

        del (
            direct_big,
            true_big,
        )

    return {
        "feature":
            np.concatenate(
                features,
                axis=0,
            ),
        "abc":
            np.concatenate(
                abcs,
                axis=0,
            ),
        "anchor":
            np.concatenate(
                anchors_out,
                axis=0,
            ),
        "channel":
            np.concatenate(
                channels_out,
                axis=0,
            ),
    }


In [ ]:

def oof_cache_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "oof"
        ]
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_oof.npz"
        )
    )


def val_cache_path(
    name,
    horizon,
):
    return (
        DIRS[
            "validation"
        ]
        / (
            f"{name}_H{horizon}_"
            "validation.npz"
        )
    )


def build_oof_fold(
    data,
    horizon,
    fold,
    p0,
    p1,
    fixed_epochs,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    out_path = oof_cache_path(
        name,
        horizon,
        fold,
    )

    if (
        out_path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            out_path
        )

        print(
            "Loaded OOF cache:",
            out_path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    prefix = int(
        p0
        * data[
            "train_end"
        ]
    )

    oof_end = int(
        p1
        * data[
            "train_end"
        ]
    )

    z, prefix_scaler = prefix_normalize(
        data[
            "raw"
        ],
        prefix,
    )

    direct_model, _ = (
        train_fold_direct(
            CURRENT_BACKBONE,
            horizon,
            z,
            prefix,
            fixed_epochs,
            fold,
        )
    )

    retriever, _ = load_frozen_retriever(
        fold_retriever_ckpt_path(
            name,
            horizon,
            fold,
        )
    )

    memory = build_memory(
        z,
        C,
        prefix,
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        (
            f"F{fold}_"
            f"prefix{prefix}"
        ),
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        prefix,
        oof_end,
        horizon,
        stride=
            OOF_ANCHOR_STRIDE,
    )

    print(
        f"OOF {name} H={horizon} F{fold}: "
        f"prefix={prefix}, "
        f"end={oof_end}, "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        z,
        anchors,
    )

    np.savez_compressed(
        out_path,
        **out,
    )

    del (
        direct_model,
        retriever,
        memory,
        memory_gpu_obj,
        prefix_scaler,
        z,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


def build_validation_cache(
    data,
    horizon,
    direct_model,
    retriever,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    path = val_cache_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            path
        )

        print(
            "Loaded validation cache:",
            path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "train_end"
        ],
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        "validation_train_memory",
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        data[
            "train_end"
        ],
        data[
            "val_end"
        ],
        horizon,
        stride=1,
    )

    print(
        f"Validation {name} H={horizon}: "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        data[
            "z"
        ],
        anchors,
    )

    np.savez_compressed(
        path,
        **out,
    )

    del (
        memory,
        memory_gpu_obj,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


In [ ]:

def fit_feature_scaler(
    x,
):
    median = np.median(
        x,
        axis=0,
    ).astype(
        np.float32
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75
        - q25
    ).astype(
        np.float32
    )

    iqr = np.where(
        iqr < 1e-5,
        1.0,
        iqr,
    ).astype(
        np.float32
    )

    return (
        median,
        iqr,
    )


def scale_features(
    x,
    median,
    iqr,
):
    return np.clip(
        (
            x
            - median
        )
        / iqr,
        -8.0,
        8.0,
    ).astype(
        np.float32
    )


def gate_loss(
    alpha,
    abc,
):
    return (
        abc[
            :,
            0
        ]
        + 2.0
        * alpha
        * abc[
            :,
            1
        ]
        + alpha
        * alpha
        * abc[
            :,
            2
        ]
    ).mean()


def gate_checkpoint_path(
    name,
    horizon,
):
    return (
        DIRS[
            "gate"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate.pt"
        )
    )


def train_gate_epoch(
    model,
    optimizer,
    x,
    abc,
    rng,
):
    model.train()

    order = rng.permutation(
        len(
            x
        )
    )

    losses = []

    for i in range(
        0,
        len(
            order
        ),
        GATE_BATCH,
    ):
        ids = order[
            i:
            i+GATE_BATCH
        ]

        xt = torch.from_numpy(
            x[
                ids
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                ids
            ]
        ).to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        alpha = model(
            xt
        )

        loss = gate_loss(
            alpha,
            at,
        )

        loss.backward()
        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

        del (
            xt,
            at,
            alpha,
            loss,
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def evaluate_gate(
    model,
    x,
    abc,
):
    model.eval()

    total = 0.0
    n = 0
    alpha_sum = 0.0

    for i in range(
        0,
        len(
            x
        ),
        GATE_BATCH,
    ):
        xt = torch.from_numpy(
            x[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        alpha = model(
            xt
        )

        each = (
            at[
                :,
                0
            ]
            + 2.0
            * alpha
            * at[
                :,
                1
            ]
            + alpha
            * alpha
            * at[
                :,
                2
            ]
        )

        total += float(
            each.sum()
        )

        n += len(
            alpha
        )

        alpha_sum += float(
            alpha.sum()
        )

        del (
            xt,
            at,
            alpha,
            each,
        )

    return (
        total
        / n,
        alpha_sum
        / n,
    )


def train_crossfit_gate(
    name,
    horizon,
    oof_x,
    oof_abc,
    val_x,
    val_abc,
):
    path = gate_checkpoint_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = CrossFitAdaptiveGate().to(
            DEVICE
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded gate:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    median, iqr = fit_feature_scaler(
        oof_x
    )

    train_x = scale_features(
        oof_x,
        median,
        iqr,
    )

    valid_x = scale_features(
        val_x,
        median,
        iqr,
    )

    seed = (
        CROSSFIT_SEED
        + horizon
        * 3000
        + sum(
            map(
                ord,
                name,
            )
        )
    )

    set_seed(
        seed
    )

    model = CrossFitAdaptiveGate().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    best = float(
        "inf"
    )

    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(
        1,
        GATE_MAX_EPOCHS
        + 1,
    ):
        train_mse = train_gate_epoch(
            model,
            optimizer,
            train_x,
            oof_abc,
            rng,
        )

        val_mse, mean_alpha = evaluate_gate(
            model,
            valid_x,
            val_abc,
        )

        history.append({
            "Epoch":
                epoch,
            "OOFTrainMSE":
                train_mse,
            "ValMSE":
                val_mse,
            "ValMeanAlpha":
                mean_alpha,
        })

        if (
            val_mse
            < best
            - 1e-10
        ):
            best = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"Gate {name:11s} H={horizon:3d} "
            f"ep={epoch:02d} "
            f"OOF={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"alpha={mean_alpha:.3f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if (
            wait
            >= GATE_PATIENCE
        ):
            break

    # Reinitialize and fit only on OOF for the
    # validation-selected number of epochs.
    set_seed(
        seed
    )

    final = CrossFitAdaptiveGate().to(
        DEVICE
    )

    final_opt = torch.optim.AdamW(
        final.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    final_rng = np.random.default_rng(
        seed
        + 2
    )

    for _ in range(
        best_epoch
    ):
        train_gate_epoch(
            final,
            final_opt,
            train_x,
            oof_abc,
            final_rng,
        )

    final.eval()

    ckpt = {
        "BestEpoch":
            best_epoch,
        "BestValMSE":
            best,
        "FeatureMedian":
            median,
        "FeatureIQR":
            iqr,
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in final.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        DIRS[
            "history"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate_history.csv"
        ),
        index=False,
    )

    return (
        final,
        ckpt,
    )


def mse_scalar(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = float(
        alpha
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_scalar(
    abc,
):
    rows = []

    best_alpha = None
    best_mse = float(
        "inf"
    )

    for alpha in ALPHA_GRID:
        mse = mse_scalar(
            abc,
            alpha,
        )

        rows.append({
            "Alpha":
                float(
                    alpha
                ),
            "MSE":
                mse,
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_alpha = float(
                alpha
            )

    return (
        best_alpha,
        pd.DataFrame(
            rows
        ),
    )


@torch.no_grad()
def gate_alpha(
    model,
    ckpt,
    x,
):
    sx = scale_features(
        x,
        ckpt[
            "FeatureMedian"
        ],
        ckpt[
            "FeatureIQR"
        ],
    )

    outputs = []

    for i in range(
        0,
        len(
            sx
        ),
        GATE_BATCH,
    ):
        t = torch.from_numpy(
            sx[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        outputs.append(
            model(
                t
            ).cpu()
            .numpy()
        )

        del t

    return np.concatenate(
        outputs
    ).astype(
        np.float32
    )


def mse_pair(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = np.asarray(
        alpha,
        dtype=np.float64,
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_lambda(
    abc,
    gate_alpha_values,
    scalar_alpha,
):
    rows = []

    best_lambda = None
    best_mse = float(
        "inf"
    )

    for lmb in LAMBDA_GRID:
        alpha = (
            (
                1.0
                - float(
                    lmb
                )
            )
            * scalar_alpha
            + float(
                lmb
            )
            * gate_alpha_values
        )

        mse = mse_pair(
            abc,
            alpha,
        )

        rows.append({
            "Lambda":
                float(
                    lmb
                ),
            "MSE":
                mse,
            "MeanAlpha":
                float(
                    alpha.mean()
                ),
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_lambda = float(
                lmb
            )

    return (
        best_lambda,
        pd.DataFrame(
            rows
        ),
    )


In [ ]:

def empty_stat():
    return {
        "sse":
            0.0,
        "sae":
            0.0,
        "n":
            0,
    }


def update_stat(
    stat,
    pred,
    true,
):
    e = (
        pred
        - true
    )

    stat[
        "sse"
    ] += float(
        (
            e
            * e
        ).sum()
    )

    stat[
        "sae"
    ] += float(
        e.abs().sum()
    )

    stat[
        "n"
    ] += e.numel()


def finish_stat(
    stat,
):
    return (
        stat[
            "sse"
        ]
        / stat[
            "n"
        ],
        stat[
            "sae"
        ]
        / stat[
            "n"
        ],
    )


def oracle_alpha_and_prediction(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    alpha = torch.clamp(
        -(
            e
            * delta
        ).sum(
            dim=1
        )
        / (
            (
                delta
                * delta
            ).sum(
                dim=1
            )
            + 1e-8
        ),
        0.0,
        1.0,
    )

    pred = (
        direct
        + alpha[
            :,
            None
        ]
        * delta
    )

    return (
        alpha,
        pred,
    )


@torch.no_grad()
def test_evaluate(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    direct_block = DIRECT_ANCHOR_BLOCK[
        CURRENT_BACKBONE
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    anchors = eval_anchors(
        data[
            "val_end"
        ],
        data[
            "test_end"
        ],
        horizon,
        stride=1,
    )

    keys = [
        "Direct",
        "Retrieval",
        "Scalar",
        "RawAdaptive",
        "ShrinkAdaptive",
        "Oracle",
    ]

    stats = {
        key:
            empty_stat()
        for key in keys
    }

    anchor_mse = {
        key:
            []
        for key in keys
    }

    channel_sse_direct = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_sse_shrink = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_count = np.zeros(
        C,
        dtype=np.int64,
    )

    raw_alpha_sum = 0.0
    shrink_alpha_sum = 0.0
    oracle_alpha_sum = 0.0
    oracle_positive = 0
    n_pairs = 0

    processed = 0

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                data[
                    "z"
                ],
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        # anchor-level MSE accumulators within this direct block
        block_anchor_sums = {
            key:
                torch.zeros(
                    len(
                        a_big
                    ),
                    device=DEVICE,
                    dtype=torch.float64,
                )
            for key in keys
        }

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                data[
                    "z"
                ],
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            direct = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            true = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            scalar = (
                direct
                + scalar_alpha
                * (
                    retrieval
                    - direct
                )
            )

            features = gate_features(
                r,
                retrieval,
                direct,
            ).cpu().numpy().astype(
                np.float32
            )

            scaled = scale_features(
                features,
                gate_ckpt[
                    "FeatureMedian"
                ],
                gate_ckpt[
                    "FeatureIQR"
                ],
            )

            gate_alpha_values = gate(
                torch.from_numpy(
                    scaled
                ).to(
                    DEVICE
                )
            )

            shrink_alpha_values = (
                (
                    1.0
                    - shrink_lambda
                )
                * scalar_alpha
                + shrink_lambda
                * gate_alpha_values
            )

            raw_adaptive = (
                direct
                + gate_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            shrink_adaptive = (
                direct
                + shrink_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            (
                oracle_alpha,
                oracle,
            ) = oracle_alpha_and_prediction(
                direct,
                retrieval,
                true,
            )

            predictions = {
                "Direct":
                    direct,
                "Retrieval":
                    retrieval,
                "Scalar":
                    scalar,
                "RawAdaptive":
                    raw_adaptive,
                "ShrinkAdaptive":
                    shrink_adaptive,
                "Oracle":
                    oracle,
            }

            for key, pred in predictions.items():
                update_stat(
                    stats[
                        key
                    ],
                    pred,
                    true,
                )

                per_anchor = (
                    (
                        (
                            pred
                            - true
                        )
                        ** 2
                    )
                    .reshape(
                        A,
                        C,
                        horizon,
                    )
                    .mean(
                        dim=(
                            1,
                            2,
                        )
                    )
                    .double()
                )

                block_anchor_sums[
                    key
                ][
                    inner:
                    inner+A
                ] = per_anchor

            direct_e2 = (
                (
                    direct
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            shrink_e2 = (
                (
                    shrink_adaptive
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            channel_sse_direct += (
                direct_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_sse_shrink += (
                shrink_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_count += (
                A
                * horizon
            )

            raw_alpha_sum += float(
                gate_alpha_values.sum()
            )

            shrink_alpha_sum += float(
                shrink_alpha_values.sum()
            )

            oracle_alpha_sum += float(
                oracle_alpha.sum()
            )

            oracle_positive += int(
                (
                    oracle_alpha
                    > 0.01
                ).sum()
            )

            n_pairs += len(
                gate_alpha_values
            )

            del (
                r,
                retrieval,
                direct,
                true,
                scalar,
                features,
                scaled,
                gate_alpha_values,
                shrink_alpha_values,
                raw_adaptive,
                shrink_adaptive,
                oracle_alpha,
                oracle,
                predictions,
                direct_e2,
                shrink_e2,
            )

        for key in keys:
            anchor_mse[
                key
            ].extend(
                block_anchor_sums[
                    key
                ].cpu()
                .numpy()
                .astype(
                    np.float32
                )
                .tolist()
            )

        processed += len(
            a_big
        )

        if (
            processed
            == len(
                a_big
            )
            or processed
            % 500
            < len(
                a_big
            )
            or processed
            == len(
                anchors
            )
        ):
            print(
                f"  test anchors "
                f"{processed}/{len(anchors)}"
            )

        del (
            direct_big,
            true_big,
            block_anchor_sums,
        )

    return {
        "anchors":
            anchors,
        "metrics": {
            key:
                finish_stat(
                    value
                )
            for key, value
            in stats.items()
        },
        "anchor_mse": {
            key:
                np.asarray(
                    value,
                    dtype=np.float32,
                )
            for key, value
            in anchor_mse.items()
        },
        "channel_direct_mse":
            channel_sse_direct
            / channel_count,
        "channel_shrink_mse":
            channel_sse_shrink
            / channel_count,
        "raw_mean_alpha":
            raw_alpha_sum
            / n_pairs,
        "shrink_mean_alpha":
            shrink_alpha_sum
            / n_pairs,
        "oracle_mean_alpha":
            oracle_alpha_sum
            / n_pairs,
        "oracle_positive_fraction":
            oracle_positive
            / n_pairs,
    }


In [ ]:

def moving_block_bootstrap(
    difference,
    n_boot=5000,
    block=24,
    seed=222222,
):
    x = np.asarray(
        difference,
        dtype=np.float64,
    )

    n = len(
        x
    )

    L = min(
        block,
        n,
    )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = int(
        np.ceil(
            n
            / L
        )
    )

    max_start = max(
        1,
        n
        - L
        + 1,
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):
        starts = rng.integers(
            0,
            max_start,
            size=n_blocks,
        )

        sample = np.concatenate(
            [
                x[
                    s:
                    s+L
                ]
                for s in starts
            ]
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "MeanImprovement":
            float(
                x.mean()
            ),
        "CI_Low":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),
        "CI_High":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),
    }


In [ ]:

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.is_file()
    )
    else pd.DataFrame()
)

summary_rows = (
    existing.to_dict(
        "records"
    )
    if len(
        existing
    )
    else []
)

bootstrap_existing = (
    pd.read_csv(
        BOOTSTRAP_PATH
    )
    if (
        RESUME
        and BOOTSTRAP_PATH.is_file()
    )
    else pd.DataFrame()
)

bootstrap_rows = (
    bootstrap_existing.to_dict(
        "records"
    )
    if len(
        bootstrap_existing
    )
    else []
)


def completed(
    backbone,
    horizon,
):
    if not len(
        existing
    ):
        return False

    return bool(
        (
            (
                existing[
                    "Backbone"
                ]
                == backbone
            )
            & (
                existing[
                    "Horizon"
                ]
                == horizon
            )
        ).any()
    )


for backbone in BACKBONES:
    CURRENT_BACKBONE = (
        backbone
    )

    DIRS = backbone_dirs(
        backbone
    )

    activate_backbone(
        backbone
    )

    for horizon in HORIZONS:
        if completed(
            backbone,
            horizon,
        ):
            print(
                f"SKIP completed | "
                f"{backbone} H={horizon}"
            )
            continue

        print(
            "\n"
            + "="
            * 150
        )

        print(
            f"EXPERIMENT 46B | SOLAR | "
            f"{backbone} | H={horizon}"
        )

        print(
            "="
            * 150
        )

        t_condition = time.time()

        direct_model, direct_ckpt = (
            train_or_load_full_direct(
                backbone,
                horizon,
            )
        )

        retriever, retriever_ckpt = (
            load_frozen_retriever(
                full_retriever_ckpt_path(
                    "Solar",
                    horizon,
                )
            )
        )

        # ---------------------------------------------------------
        # Chronological OOF.
        # ---------------------------------------------------------
        oof_parts = []

        for fold, (
            p0,
            p1,
        ) in enumerate(
            FOLDS,
            start=1,
        ):
            out = build_oof_fold(
                DATA[
                    "Solar"
                ],
                horizon,
                fold,
                p0,
                p1,
                int(
                    direct_ckpt[
                        "BestEpoch"
                    ]
                ),
            )

            oof_parts.append(
                out
            )

        oof_x = np.concatenate(
            [
                p[
                    "feature"
                ]
                for p in oof_parts
            ],
            axis=0,
        ).astype(
            np.float32
        )

        oof_abc = np.concatenate(
            [
                p[
                    "abc"
                ]
                for p in oof_parts
            ],
            axis=0,
        ).astype(
            np.float32
        )

        # ---------------------------------------------------------
        # Validation-only integration selection.
        # ---------------------------------------------------------
        val = build_validation_cache(
            DATA[
                "Solar"
            ],
            horizon,
            direct_model,
            retriever,
        )

        gate, gate_ckpt = (
            train_crossfit_gate(
                "Solar",
                horizon,
                oof_x,
                oof_abc,
                val[
                    "feature"
                ],
                val[
                    "abc"
                ],
            )
        )

        scalar_alpha, scalar_curve = (
            choose_scalar(
                val[
                    "abc"
                ]
            )
        )

        val_gate_alpha = gate_alpha(
            gate,
            gate_ckpt,
            val[
                "feature"
            ],
        )

        shrink_lambda, lambda_curve = (
            choose_lambda(
                val[
                    "abc"
                ],
                val_gate_alpha,
                scalar_alpha,
            )
        )

        scalar_val_mse = mse_scalar(
            val[
                "abc"
            ],
            scalar_alpha,
        )

        raw_val_mse = mse_pair(
            val[
                "abc"
            ],
            val_gate_alpha,
        )

        final_val_alpha = (
            (
                1.0
                - shrink_lambda
            )
            * scalar_alpha
            + shrink_lambda
            * val_gate_alpha
        )

        shrink_val_mse = mse_pair(
            val[
                "abc"
            ],
            final_val_alpha,
        )

        calibration = (
            lambda_curve.copy()
        )

        calibration[
            "Backbone"
        ] = backbone

        calibration[
            "Dataset"
        ] = "Solar"

        calibration[
            "Horizon"
        ] = horizon

        calibration[
            "ScalarAlpha"
        ] = scalar_alpha

        calibration.to_csv(
            DIRS[
                "calibration"
            ]
            / (
                f"Solar_H{horizon}_"
                "lambda_curve.csv"
            ),
            index=False,
        )

        print(
            f"Validation | alpha0={scalar_alpha:.1f} | "
            f"lambda={shrink_lambda:.2f} | "
            f"scalar={scalar_val_mse:.6f} | "
            f"rawGate={raw_val_mse:.6f} | "
            f"shrink={shrink_val_mse:.6f}"
        )

        # ---------------------------------------------------------
        # Test memory: train + validation only.
        # ---------------------------------------------------------
        test_memory = build_memory(
            z_full,
            n_channels,
            val_end,
            horizon,
        )

        test_memory_gpu = memory_gpu_cached(
            "Solar",
            horizon,
            "test_trainval_memory",
            retriever,
            test_memory,
            n_channels,
        )

        test = test_evaluate(
            DATA[
                "Solar"
            ],
            horizon,
            direct_model,
            retriever,
            test_memory_gpu,
            gate,
            gate_ckpt,
            scalar_alpha,
            shrink_lambda,
        )

        metrics = test[
            "metrics"
        ]

        direct_mse, direct_mae = metrics[
            "Direct"
        ]

        retrieval_mse, retrieval_mae = metrics[
            "Retrieval"
        ]

        scalar_mse, scalar_mae = metrics[
            "Scalar"
        ]

        raw_mse, raw_mae = metrics[
            "RawAdaptive"
        ]

        ours_mse, ours_mae = metrics[
            "ShrinkAdaptive"
        ]

        oracle_mse, oracle_mae = metrics[
            "Oracle"
        ]

        # ---------------------------------------------------------
        # Moving-block bootstrap.
        # Positive = Direct MSE > Ours MSE.
        # ---------------------------------------------------------
        diff = (
            test[
                "anchor_mse"
            ][
                "Direct"
            ]
            - test[
                "anchor_mse"
            ][
                "ShrinkAdaptive"
            ]
        )

        boot = moving_block_bootstrap(
            diff,
            n_boot=
                BOOTSTRAP_REPLICATES,
            block=
                BOOTSTRAP_BLOCK_LEN,
            seed=
                BOOTSTRAP_SEED
                + horizon
                + 10000
                * BACKBONES.index(
                    backbone
                ),
        )

        bootstrap_rows = [
            r
            for r in bootstrap_rows
            if not (
                r.get(
                    "Backbone"
                )
                == backbone
                and int(
                    r.get(
                        "Horizon",
                        -1,
                    )
                )
                == horizon
            )
        ]

        bootstrap_rows.append({
            "Backbone":
                backbone,
            "Dataset":
                "Solar",
            "Horizon":
                horizon,
            "Comparison":
                "Direct-ShrinkAdaptive",
            "MeanImprovement":
                boot[
                    "MeanImprovement"
                ],
            "CI_Low":
                boot[
                    "CI_Low"
                ],
            "CI_High":
                boot[
                    "CI_High"
                ],
            "SignificantPositive":
                bool(
                    boot[
                        "CI_Low"
                    ]
                    > 0
                ),
            "SignificantNegative":
                bool(
                    boot[
                        "CI_High"
                    ]
                    < 0
                ),
            "BlockLen":
                BOOTSTRAP_BLOCK_LEN,
            "Replicates":
                BOOTSTRAP_REPLICATES,
        })

        # ---------------------------------------------------------
        # Channel diagnostics.
        # ---------------------------------------------------------
        channel_df = pd.DataFrame({
            "Channel":
                np.arange(
                    n_channels
                ),
            "Direct_MSE":
                test[
                    "channel_direct_mse"
                ],
            "Ours_MSE":
                test[
                    "channel_shrink_mse"
                ],
        })

        channel_df[
            "Gain_pct"
        ] = (
            100.0
            * (
                channel_df[
                    "Direct_MSE"
                ]
                - channel_df[
                    "Ours_MSE"
                ]
            )
            / channel_df[
                "Direct_MSE"
            ]
        )

        channel_df.to_csv(
            DIRS[
                "channel"
            ]
            / (
                f"Solar_H{horizon}_"
                "channel.csv"
            ),
            index=False,
        )

        improved_channel_fraction = float(
            (
                channel_df[
                    "Ours_MSE"
                ]
                < channel_df[
                    "Direct_MSE"
                ]
            ).mean()
        )

        oracle_headroom = (
            100.0
            * (
                direct_mse
                - oracle_mse
            )
            / direct_mse
        )

        row = {
            "Backbone":
                backbone,
            "Dataset":
                "Solar",
            "Horizon":
                horizon,
            "DirectSeqLen":
                direct_seq_len(
                    backbone
                ),
            "RetrievalSeqLen":
                RET_SEQ_LEN,

            "Direct_MSE":
                direct_mse,
            "ShrinkAdaptive_MSE":
                ours_mse,
            "Direct_MAE":
                direct_mae,
            "ShrinkAdaptive_MAE":
                ours_mae,

            "Retrieval_MSE":
                retrieval_mse,
            "Retrieval_MAE":
                retrieval_mae,
            "Scalar_MSE":
                scalar_mse,
            "Scalar_MAE":
                scalar_mae,
            "RawAdaptive_MSE":
                raw_mse,
            "RawAdaptive_MAE":
                raw_mae,
            "Oracle_MSE":
                oracle_mse,
            "Oracle_MAE":
                oracle_mae,

            "ScalarAlpha":
                scalar_alpha,
            "ShrinkLambda":
                shrink_lambda,
            "RawMeanAlpha":
                test[
                    "raw_mean_alpha"
                ],
            "ShrinkMeanAlpha":
                test[
                    "shrink_mean_alpha"
                ],

            "MSEGain_pct":
                100.0
                * (
                    direct_mse
                    - ours_mse
                )
                / direct_mse,

            "MAEGain_pct":
                100.0
                * (
                    direct_mae
                    - ours_mae
                )
                / direct_mae,

            "OracleHeadroomFromDirect_pct":
                oracle_headroom,

            "ImprovedChannelFraction":
                improved_channel_fraction,

            "GateBestEpoch":
                gate_ckpt[
                    "BestEpoch"
                ],

            "DirectBestEpoch":
                direct_ckpt[
                    "BestEpoch"
                ],

            "OOFPairs":
                len(
                    oof_x
                ),

            "TestAnchors":
                len(
                    test[
                        "anchors"
                    ]
                ),

            "TestMemoryPerChannel":
                test_memory[
                    "M"
                ],

            "RuntimeMinutes":
                (
                    time.time()
                    - t_condition
                )
                / 60.0,
        }

        summary_rows = [
            r
            for r in summary_rows
            if not (
                r.get(
                    "Backbone"
                )
                == backbone
                and int(
                    r.get(
                        "Horizon",
                        -1,
                    )
                )
                == horizon
            )
        ]

        summary_rows.append(
            row
        )

        np.savez_compressed(
            DIRS[
                "paired"
            ]
            / (
                f"Solar_H{horizon}_"
                "anchor_mse.npz"
            ),
            Anchors=
                test[
                    "anchors"
                ],
            Direct=
                test[
                    "anchor_mse"
                ][
                    "Direct"
                ],
            Retrieval=
                test[
                    "anchor_mse"
                ][
                    "Retrieval"
                ],
            Scalar=
                test[
                    "anchor_mse"
                ][
                    "Scalar"
                ],
            RawAdaptive=
                test[
                    "anchor_mse"
                ][
                    "RawAdaptive"
                ],
            ShrinkAdaptive=
                test[
                    "anchor_mse"
                ][
                    "ShrinkAdaptive"
                ],
            Oracle=
                test[
                    "anchor_mse"
                ][
                    "Oracle"
                ],
        )

        pd.DataFrame(
            summary_rows
        ).sort_values(
            [
                "Backbone",
                "Horizon",
            ]
        ).to_csv(
            SUMMARY_PATH,
            index=False,
        )

        pd.DataFrame(
            bootstrap_rows
        ).sort_values(
            [
                "Backbone",
                "Horizon",
            ]
        ).to_csv(
            BOOTSTRAP_PATH,
            index=False,
        )

        print(
            "\nFINAL | "
            f"{backbone} H={horizon} | "
            f"Direct={direct_mse:.6f} | "
            f"Ours={ours_mse:.6f} | "
            f"Gain={row['MSEGain_pct']:+.3f}% | "
            f"CI=[{boot['CI_Low']:.6f}, "
            f"{boot['CI_High']:.6f}]"
        )

        del (
            direct_model,
            retriever,
            gate,
            test_memory,
            test_memory_gpu,
            test,
            val,
            oof_parts,
            oof_x,
            oof_abc,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:

summary = pd.read_csv(
    SUMMARY_PATH
)

expected = pd.MultiIndex.from_product(
    [
        BACKBONES,
        HORIZONS,
    ],
    names=[
        "Backbone",
        "Horizon",
    ],
).to_frame(
    index=False
)

present = summary[
    [
        "Backbone",
        "Horizon",
    ]
].drop_duplicates()

missing = expected.merge(
    present,
    on=[
        "Backbone",
        "Horizon",
    ],
    how="left",
    indicator=True,
)

missing = missing[
    missing[
        "_merge"
    ]
    == "left_only"
]

if len(
    missing
):
    display(
        missing
    )

    raise RuntimeError(
        "Not all requested Solar DLinear conditions are complete."
    )

if len(
    summary
) != len(BACKBONES) * len(HORIZONS):
    raise RuntimeError(
        f"Expected the requested number of summary rows, found {len(summary)}."
    )

print(
    "PASS: all requested Solar DLinear conditions are complete."
)

display(
    summary.sort_values(
        [
            "Backbone",
            "Horizon",
        ]
    )
)


In [ ]:

boot_df = pd.read_csv(
    BOOTSTRAP_PATH
)

boot_key = boot_df[
    boot_df[
        "Comparison"
    ]
    == "Direct-ShrinkAdaptive"
][
    [
        "Backbone",
        "Horizon",
        "CI_Low",
        "CI_High",
        "SignificantPositive",
        "SignificantNegative",
    ]
]

evidence = summary.merge(
    boot_key,
    on=[
        "Backbone",
        "Horizon",
    ],
    how="left",
)

evidence[
    "MSE_Win"
] = (
    evidence[
        "ShrinkAdaptive_MSE"
    ]
    < evidence[
        "Direct_MSE"
    ]
)

evidence[
    "MAE_Win"
] = (
    evidence[
        "ShrinkAdaptive_MAE"
    ]
    < evidence[
        "Direct_MAE"
    ]
)

backbone_summary = (
    evidence
    .groupby(
        "Backbone",
        as_index=False,
    )
    .agg(
        Conditions=(
            "Horizon",
            "size",
        ),
        MSE_Wins=(
            "MSE_Win",
            "sum",
        ),
        SignificantWins=(
            "SignificantPositive",
            "sum",
        ),
        SignificantLosses=(
            "SignificantNegative",
            "sum",
        ),
        MeanMSEGain_pct=(
            "MSEGain_pct",
            "mean",
        ),
        MAE_Wins=(
            "MAE_Win",
            "sum",
        ),
        MeanMAEGain_pct=(
            "MAEGain_pct",
            "mean",
        ),
        MeanOracleHeadroom_pct=(
            "OracleHeadroomFromDirect_pct",
            "mean",
        ),
        MeanShrinkAlpha=(
            "ShrinkMeanAlpha",
            "mean",
        ),
    )
)

display(
    backbone_summary
)

backbone_summary.to_csv(
    ROOT
    / "solar_backbone_summary.csv",
    index=False,
)


task_summary = (
    evidence
    .groupby(
        "Horizon",
        as_index=False,
    )
    .agg(
        BackboneWins=(
            "MSE_Win",
            "sum",
        ),
        SignificantBackboneWins=(
            "SignificantPositive",
            "sum",
        ),
        SignificantBackboneLosses=(
            "SignificantNegative",
            "sum",
        ),
        MeanMSEGain_pct=(
            "MSEGain_pct",
            "mean",
        ),
    )
)

display(
    task_summary
)

task_summary.to_csv(
    ROOT
    / "solar_horizon_three_backbone_summary.csv",
    index=False,
)


In [ ]:

print(
    "Experiment root:",
    ROOT,
)

for p in sorted(
    ROOT.rglob(
        "*"
    )
):
    if p.is_file():
        print(
            " -",
            p.relative_to(
                ROOT
            )
        )
